**COMPARABLE SALES**

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("train.csv")
train,test = train_test_split(df, test_size=0.2, random_state=42)

def find_comps(house, pool, n=5):
  similar = pool[(pool["Neighborhood"] == house["Neighborhood"]) &
                 ((pool["OverallQual"] - house["OverallQual"]).abs() <= 1)].copy()
  similar["size_gap"] = (similar["GrLivArea"] - house["GrLivArea"]).abs()
  return similar.sort_values("size_gap").head(n)

house = test.iloc[0]
print(house[["Neighborhood", "GrLivArea", "OverallQual", "YearBuilt", "SalePrice"]])
print()

comps = find_comps(house, train)
print(comps[["Neighborhood", "GrLivArea", "OverallQual", "YearBuilt", "SalePrice"]])
print()
print("Comps prediction (average of their prices):", round(comps["SalePrice"].mean()))
print("Actual price:", house["SalePrice"])



Neighborhood    Sawyer
GrLivArea         1068
OverallQual          6
YearBuilt         1963
SalePrice       154500
Name: 892, dtype: object

     Neighborhood  GrLivArea  OverallQual  YearBuilt  SalePrice
24         Sawyer       1060            5       1968     154000
957        Sawyer       1057            5       1962     132000
116        Sawyer       1092            5       1962     139000
10         Sawyer       1040            5       1965     129500
1081       Sawyer       1040            5       1963     133000

Comps prediction (average of their prices): 137500
Actual price: 154500


*The five comps are very similar to the house (same neighborhood, 1,040 to 1,092 sq ft, built 1962 to 1968), and their average price (137,500) is about 11% below the actual price (154,500). All five comps have quality 5, while the house has quality 6, so an appraiser would adjust the estimate upward.*

**COMPARE COMPS WITH MODEL ON EVERY TEST HOUSE**

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

def comps_predict(house, pool, n=5):
    comps = find_comps(house, pool, n)
    if len(comps) < 3:
        return float("nan")
    return comps["SalePrice"].mean()

test = test.copy()
test["comps_pred"] = test.apply(lambda r: comps_predict(r, train), axis=1)

# Linear model: five numeric features + neighborhood (your Day 17 model)
base_cols = ["GrLivArea", "OverallQual", "GarageCars", "YearBuilt", "TotalBsmtSF"]
hood_train = pd.get_dummies(train["Neighborhood"], prefix="Hood", dtype=int)
hood_test = pd.get_dummies(test["Neighborhood"], prefix="Hood", dtype=int)
hood_test = hood_test.reindex(columns=hood_train.columns, fill_value=0)
X_tr = pd.concat([train[base_cols], hood_train], axis=1)
X_te = pd.concat([test[base_cols], hood_test], axis=1)
model = LinearRegression().fit(X_tr, train["SalePrice"])
test["model_pred"] = model.predict(X_te)

has = test["comps_pred"].notna()
print("Test houses with at least 3 comps:", has.sum(), "of", len(test))
print("Comps MAE:", round(mean_absolute_error(test.loc[has, "SalePrice"], test.loc[has, "comps_pred"])))
print("Model MAE (same houses):", round(mean_absolute_error(test.loc[has, "SalePrice"], test.loc[has, "model_pred"])))

Test houses with at least 3 comps: 289 of 292
Comps MAE: 22585
Model MAE (same houses): 22391


In [6]:
test["blend_pred"] = (test["comps_pred"] + test["model_pred"]) / 2
print("Blend MAE:", round(mean_absolute_error(test.loc[has, "SalePrice"], test.loc[has, "blend_pred"])))

test["comps_err"] = (test["comps_pred"] - test["SalePrice"]).abs()
test["model_err"] = (test["model_pred"] - test["SalePrice"]).abs()
cols = ["Neighborhood", "GrLivArea", "OverallQual", "SalePrice", "comps_pred", "model_pred"]
test.loc[has].sort_values("comps_err", ascending=False)[cols].head(5).round()

Blend MAE: 20498


,Neighborhood,GrLivArea,OverallQual,SalePrice,comps_pred,model_pred
898,NridgHt,2364,9,611657,395510.0,365099.0
691,NoRidge,4316,10,755000,542417.0,469014.0
1046,StoneBr,2868,9,556581,419851.0,384889.0
56,Somerst,1739,8,172500,292271.0,234731.0
478,CollgCr,1718,8,297000,194800.0,257327.0


*Averaging the comps prediction and the linear model’s prediction gives an average error of 20,498, about 8% lower than either method alone (22,585 and 22,391) and about 67% below the baseline of 62,576. The houses both methods get most wrong are the most expensive ones (quality 9 or 10), which both underpredict, so blending helps on typical houses and not on luxury homes. This is one test split, so a repeated comparison would make the result firmer.*